In [52]:
print("Hybrid Recommendation Demo Notebook")

Hybrid Recommendation Demo Notebook


# Hybrid Recommendation System Demo

This notebook provides a simple demo of a hybrid recommendation system using sample product recommendation data.

It covers:

- Content-Based Filtering
- Collaborative Filtering
- Hybrid Recommendation Logic

The goal is to show how multiple recommendation methods can be combined to produce more balanced product recommendations.


In [53]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [54]:
products = pd.DataFrame({
    "Product": [
        "Wireless Keyboard",
        "Bluetooth Speaker",
        "Fitness Band",
        "Study Lamp",
        "Laptop Stand",
        "Noise Cancelling Earbuds"
    ],
    "Category": [
        "Accessories",
        "Electronics",
        "Wearable",
        "Home Office",
        "Accessories",
        "Electronics"
    ],
    "Price": [
        1800,
        3500,
        2500,
        1200,
        900,
        4500
    ],
    "Rating": [
        4.3,
        4.5,
        4.1,
        4.0,
        4.2,
        4.6
    ]
})

products


,Product,Category,Price,Rating
0,Wireless Keyboard,Accessories,1800,4.3
1,Bluetooth Speaker,Electronics,3500,4.5
2,Fitness Band,Wearable,2500,4.1
3,Study Lamp,Home Office,1200,4.0
4,Laptop Stand,Accessories,900,4.2
5,Noise Cancelling Earbuds,Electronics,4500,4.6


## Content-Based Recommendation

Content-based filtering recommends products that are similar to a selected product by comparing product features such as category, price range, and rating.


In [55]:
# Convert product categories into one-hot encoded features
encoded_categories = pd.get_dummies(
    products["Category"],
    prefix="Category"
)

# Calculate cosine similarity between products using category features
product_similarity_matrix = cosine_similarity(encoded_categories)

# Convert the similarity matrix into a DataFrame for better readability
product_similarity_df = pd.DataFrame(
    product_similarity_matrix,
    index=products["Product"],
    columns=products["Product"]
)

product_similarity_df


Product,Wireless Keyboard,Bluetooth Speaker,Fitness Band,Study Lamp,Laptop Stand,Noise Cancelling Earbuds
Product,,,,,,
Wireless Keyboard,1.0,0.0,0.0,0.0,1.0,0.0
Bluetooth Speaker,0.0,1.0,0.0,0.0,0.0,1.0
Fitness Band,0.0,0.0,1.0,0.0,0.0,0.0
Study Lamp,0.0,0.0,0.0,1.0,0.0,0.0
Laptop Stand,1.0,0.0,0.0,0.0,1.0,0.0
Noise Cancelling Earbuds,0.0,1.0,0.0,0.0,0.0,1.0


In [56]:
def recommend_content_based(product_name, top_n=3):
    if product_name not in product_similarity_df.index:
        return f"Product '{product_name}' not found in the dataset."

    similar_scores = product_similarity_df[product_name].sort_values(
        ascending=False
    )

    recommended_products = similar_scores.index[1:top_n + 1]

    return list(recommended_products)


In [57]:
recommend_content_based("Bluetooth Speaker")

['Noise Cancelling Earbuds', 'Wireless Keyboard', 'Fitness Band']

## Collaborative Filtering

Collaborative filtering recommends products based on user preferences and ratings.

In [58]:
user_ratings = pd.DataFrame({
    "User": [
        "User_1", "User_1", "User_1",
        "User_2", "User_2", "User_2",
        "User_3", "User_3", "User_3",
        "User_4", "User_4", "User_4",
        "User_5", "User_5"
    ],
    "Product": [
        "Wireless Keyboard",
        "Bluetooth Speaker",
        "Laptop Stand",

        "Bluetooth Speaker",
        "Noise Cancelling Earbuds",
        "Fitness Band",

        "Fitness Band",
        "Study Lamp",
        "Laptop Stand",

        "Wireless Keyboard",
        "Laptop Stand",
        "Noise Cancelling Earbuds",

        "Study Lamp",
        "Bluetooth Speaker"
    ],
    "Rating": [
        5, 4, 4,
        5, 5, 3,
        4, 4, 5,
        5, 4, 5,
        3, 4
    ]
})

user_ratings


,User,Product,Rating
0,User_1,Wireless Keyboard,5
1,User_1,Bluetooth Speaker,4
2,User_1,Laptop Stand,4
3,User_2,Bluetooth Speaker,5
4,User_2,Noise Cancelling Earbuds,5
5,User_2,Fitness Band,3
6,User_3,Fitness Band,4
7,User_3,Study Lamp,4
8,User_3,Laptop Stand,5
9,User_4,Wireless Keyboard,5


In [59]:
# Convert user ratings into a matrix format
# Rows represent users, columns represent products, and values represent ratings
user_item_matrix = user_ratings.pivot_table(
    index="User",
    columns="Product",
    values="Rating",
    fill_value=0
)

user_item_matrix


Product,Bluetooth Speaker,Fitness Band,Laptop Stand,Noise Cancelling Earbuds,Study Lamp,Wireless Keyboard
User,,,,,,
User_1,4.0,0.0,4.0,0.0,0.0,5.0
User_2,5.0,3.0,0.0,5.0,0.0,0.0
User_3,0.0,4.0,5.0,0.0,4.0,0.0
User_4,0.0,0.0,4.0,5.0,0.0,5.0
User_5,4.0,0.0,0.0,0.0,3.0,0.0


In [60]:
# Calculate item-item similarity based on user rating patterns
collaborative_similarity = cosine_similarity(user_item_matrix.T)

collaborative_similarity_df = pd.DataFrame(
    collaborative_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

collaborative_similarity_df


Product,Bluetooth Speaker,Fitness Band,Laptop Stand,Noise Cancelling Earbuds,Study Lamp,Wireless Keyboard
Product,,,,,,
Bluetooth Speaker,1.000000,0.397360,0.280702,0.468293,0.317888,0.374634
Fitness Band,0.397360,1.000000,0.529813,0.424264,0.640000,0.000000
Laptop Stand,0.280702,0.529813,1.000000,0.374634,0.529813,0.749269
Noise Cancelling Earbuds,0.468293,0.424264,0.374634,1.000000,0.000000,0.500000
Study Lamp,0.317888,0.640000,0.529813,0.000000,1.000000,0.000000
Wireless Keyboard,0.374634,0.000000,0.749269,0.500000,0.000000,1.000000


In [61]:
def recommend_collaborative(product_name, top_n=3):
    if product_name not in collaborative_similarity_df.index:
        return f"Product '{product_name}' not found in the collaborative similarity matrix."

    similar_scores = collaborative_similarity_df[product_name].sort_values(
        ascending=False
    )

    recommended_products = similar_scores.index[1:top_n + 1]

    return list(recommended_products)


In [62]:
recommend_collaborative("Bluetooth Speaker")

['Noise Cancelling Earbuds', 'Fitness Band', 'Wireless Keyboard']

## Hybrid Recommendation System

A hybrid recommendation system combines content-based filtering and collaborative filtering to generate more balanced and accurate product recommendations.


In [63]:
def hybrid_recommend(product_name, top_n=5):
    content_recommendations = recommend_content_based(product_name)
    collaborative_recommendations = recommend_collaborative(product_name)

    combined_recommendations = (
        content_recommendations + collaborative_recommendations
    )

    final_recommendations = list(dict.fromkeys(combined_recommendations))

    return final_recommendations[:top_n]


In [64]:
hybrid_recommend("Bluetooth Speaker")


['Noise Cancelling Earbuds', 'Wireless Keyboard', 'Fitness Band']

## Conclusion

This notebook demonstrated how different recommendation techniques can be used to suggest relevant products.

It covered:

- Content-Based Recommendation using product features
- Collaborative Filtering using user rating patterns
- Hybrid Recommendation by combining both approaches

Using Python, Pandas, and Scikit-learn, we built a simple recommendation workflow that can be extended further with larger datasets and more advanced features.
